## 5. CRISP-DM: Modeling and Evaluation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, learning_curve, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (confusion_matrix, roc_auc_score, roc_curve, accuracy_score, 
                           recall_score, precision_score, f1_score, auc, classification_report)
import seaborn as sns
from IPython.display import display, Markdown
from imblearn.over_sampling import SMOTE
import joblib
import os
import warnings

# Suppress convergence warnings for cleaner output
warnings.filterwarnings('ignore', category=UserWarning, module='sklearn')

# Settings for plotting
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

In [ ]:
# Load the prepared datasets
try:
    X_train = pd.read_csv('../data/X_train.csv')
    X_test = pd.read_csv('../data/X_test.csv')
    y_train = pd.read_csv('../data/y_train.csv')
    y_test = pd.read_csv('../data/y_test.csv')
    
    # Convert y to series (in case it was saved as DataFrame)
    if isinstance(y_train, pd.DataFrame):
        y_train = y_train.iloc[:, 0]
    if isinstance(y_test, pd.DataFrame):
        y_test = y_test.iloc[:, 0]
        
    print("Loaded prepared datasets:")
    print(f"X_train shape: {X_train.shape}")
    print(f"X_test shape: {X_test.shape}")
    print(f"y_train shape: {y_train.shape}")
    print(f"y_test shape: {y_test.shape}")
    
except FileNotFoundError:
    print("Prepared datasets not found. Loading and preparing data...")
    
    # Load original dataset
    df = pd.read_csv('../data/Cardiovascular_Disease_Dataset_1.csv')
    
    # Basic preprocessing
    X = df.drop('target', axis=1)
    y = df['target']
    
    # Encode categorical variables if needed
    X_encoded = pd.get_dummies(X, drop_first=True)
    
    # Split the data
    X_train, X_test, y_train, y_test = train_test_split(
        X_encoded, y, test_size=0.2, random_state=42, stratify=y
    )
    
    print(f"Prepared data on-the-fly:")
    print(f"X_train shape: {X_train.shape}")
    print(f"X_test shape: {X_test.shape}")

print(f"\nTarget distribution in training set:")
print(y_train.value_counts())
print(f"\nTarget distribution in test set:")
print(y_test.value_counts())

## 5.2 Model Selection and Preparation

In [ ]:
def prepare_models():
    """Prepare a dictionary of models to evaluate"""
    models = {
        'Dummy Classifier': DummyClassifier(strategy='most_frequent', random_state=42),
        'Logistic Regression': LogisticRegression(random_state=42, max_iter=5000),
        'Decision Tree': DecisionTreeClassifier(random_state=42),
        'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100),
        'Gradient Boosting': GradientBoostingClassifier(random_state=42),
        'AdaBoost': AdaBoostClassifier(random_state=42),
        # 'SVM': SVC(random_state=42, probability=True),
        'K-Nearest Neighbors': KNeighborsClassifier(),
        'Naive Bayes': GaussianNB()
    }
    return models

def train_models(models, X_train, y_train):
    """Train all models"""
    trained_models = {}
    for name, model in models.items():
        print(f"Training {name}...")
        model.fit(X_train, y_train)
        trained_models[name] = model
    return trained_models

def evaluate_models(trained_models, X_test, y_test, show_plots=True):
    """Evaluate all trained models"""
    results = {}
    
    for name, model in trained_models.items():
        print(f"\nEvaluating {name}:")
        
        # Predictions
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None
        
        # Metrics
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        roc_auc = roc_auc_score(y_test, y_prob) if y_prob is not None else 'N/A'
        
        results[name] = {
            'Accuracy': accuracy,
            'Precision': precision,
            'Recall': recall,
            'F1-Score': f1,
            'ROC AUC': roc_auc
        }
        
        print(f"  Accuracy: {accuracy:.4f}")
        print(f"  Precision: {precision:.4f}")
        print(f"  Recall: {recall:.4f}")
        print(f"  F1-Score: {f1:.4f}")
        print(f"  ROC AUC: {roc_auc:.4f}" if roc_auc != 'N/A' else f"  ROC AUC: {roc_auc}")
        
        if show_plots:
            # Confusion Matrix
            plt.figure(figsize=(6, 5))
            
            cm = confusion_matrix(y_test, y_pred)
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                       xticklabels=['No Disease', 'Disease'], 
                       yticklabels=['No Disease', 'Disease'])
            plt.title(f'Confusion Matrix - {name}')
            plt.xlabel('Predicted Label')
            plt.ylabel('True Label')
            
            plt.tight_layout()
            plt.show()
    
    return pd.DataFrame(results).T

### 5.3 Model Training and Initial Evaluation

In [ ]:
# Prepare and train models
models = prepare_models()
trained_models = train_models(models, X_train, y_train)

# Evaluate models
print("="*60)
print("INITIAL MODEL EVALUATION")
print("="*60)

results_df = evaluate_models(trained_models, X_test, y_test, show_plots=False)
display(results_df.round(4))

# Show detailed plots for ALL models
print("\n" + "="*60)
print("DETAILED EVALUATION FOR ALL MODELS")
print("="*60)

# Show detailed plots for all models
for model_name in trained_models.keys():
    model = trained_models[model_name]
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None
    
    plt.figure(figsize=(6, 5))
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
               xticklabels=['No Disease', 'Disease'], 
               yticklabels=['No Disease', 'Disease'])
    plt.title(f'Confusion Matrix - {model_name}')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    
    plt.tight_layout()
    plt.show()

### 5.4 Hyperparameter Tuning

In [ ]:
# Define parameter grids for all tunable models
param_grids = {
    'Logistic Regression': {
        'C': [0.1, 1, 10, 100],
        'penalty': ['l1', 'l2'],
        'solver': ['liblinear', 'saga']
    },
    'Decision Tree': {
        'max_depth': [3, 5, 7, 10, 15, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'criterion': ['gini', 'entropy']
    },
    'Random Forest': {
        'n_estimators': [50, 100, 200],
        'max_depth': [5, 10, 15, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    },
    'Gradient Boosting': {
        'n_estimators': [50, 100, 200],
        'learning_rate': [0.01, 0.1, 0.2],
        'max_depth': [3, 5, 7],
        'subsample': [0.8, 0.9, 1.0]
    },
    'AdaBoost': {
        'n_estimators': [50, 100, 200],
        'learning_rate': [0.5, 1.0, 1.5],
        'algorithm': ['SAMME', 'SAMME.R']
    },
    # 'SVM': {
    #     'C': [0.1, 1, 10],
    #     'kernel': ['rbf', 'poly'],
    #     'gamma': ['scale', 'auto']
    # },
    'K-Nearest Neighbors': {
        'n_neighbors': [3, 5, 7, 11, 15],
        'weights': ['uniform', 'distance'],
        'metric': ['euclidean', 'manhattan']
    }
}

In [ ]:
def hyperparameter_tuning(model, param_grid, X_train, y_train, cv=5):
    """Perform hyperparameter tuning using GridSearchCV"""
    grid_search = GridSearchCV(
        model, param_grid=param_grid, cv=cv, 
        scoring='roc_auc', verbose=1, n_jobs=-1
    )
    grid_search.fit(X_train, y_train)
    
    print(f"Best parameters: {grid_search.best_params_}")
    print(f"Best cross-validation score: {grid_search.best_score_:.4f}")
    
    return grid_search.best_estimator_

# Tune hyperparameters for all tunable models
tuned_models = {}

print("="*60)
print("HYPERPARAMETER TUNING")
print("="*60)

for model_name in param_grids.keys():
    if model_name in models:
        print(f"\nTuning {model_name}...")
        base_model = models[model_name]
        best_model = hyperparameter_tuning(base_model, param_grids[model_name], X_train, y_train)
        tuned_models[model_name] = best_model

# Add models that don't require tuning to the tuned_models dict
models_no_tuning = ['Dummy Classifier', 'Naive Bayes']
for model_name in models_no_tuning:
    if model_name in models:
        print(f"\n{model_name} - No hyperparameter tuning needed")
        tuned_models[model_name] = models[model_name]

### 5.5 Cross-Validation and Overfitting Analysis

In [ ]:
# Cross-validation scores
print("="*60)
print("CROSS-VALIDATION SCORES")
print("="*60)

cv_scores = {}
for name, model in tuned_models.items():
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='roc_auc')
    cv_scores[name] = scores
    print(f"{name}:")
    print(f"  CV ROC AUC: {scores.mean():.4f} (+/- {scores.std() * 2:.4f})")

# Decision Tree Overfitting Analysis
print("\n" + "="*60)
print("DECISION TREE OVERFITTING ANALYSIS")
print("="*60)

def plot_tree_depth_vs_performance(X_train, y_train, X_test, y_test, max_depth_range=range(1, 21)):
    """Plot decision tree performance vs depth to show overfitting"""
    train_scores = []
    test_scores = []
    depths = list(max_depth_range)
    
    for depth in depths:
        dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
        dt.fit(X_train, y_train)
        
        train_prob = dt.predict_proba(X_train)[:, 1]
        test_prob = dt.predict_proba(X_test)[:, 1]
        
        train_auc = roc_auc_score(y_train, train_prob)
        test_auc = roc_auc_score(y_test, test_prob)
        
        train_scores.append(train_auc)
        test_scores.append(test_auc)
    
    plt.figure(figsize=(10, 6))
    plt.plot(depths, train_scores, 'o-', label='Training ROC AUC', color='blue')
    plt.plot(depths, test_scores, 'o-', label='Test ROC AUC', color='red')
    plt.xlabel('Tree Depth')
    plt.ylabel('ROC AUC Score')
    plt.title('Decision Tree: ROC AUC vs Tree Depth (Overfitting Analysis)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Find optimal depth
    best_depth = depths[np.argmax(test_scores)]
    best_test_score = max(test_scores)
    plt.axvline(x=best_depth, color='green', linestyle='--', alpha=0.7, 
                label=f'Optimal Depth: {best_depth} (AUC: {best_test_score:.3f})')
    plt.legend()
    
    plt.tight_layout()
    plt.show()
    
    return depths, train_scores, test_scores, best_depth

depths, train_scores, test_scores, optimal_depth = plot_tree_depth_vs_performance(X_train, y_train, X_test, y_test)

print(f"Optimal tree depth: {optimal_depth}")
print(f"Test ROC AUC at optimal depth: {test_scores[optimal_depth-1]:.4f}")
print(f"Training ROC AUC at optimal depth: {train_scores[optimal_depth-1]:.4f}")
print(f"Overfitting gap: {train_scores[optimal_depth-1] - test_scores[optimal_depth-1]:.4f}")

### 5.6 Final Model Evaluation

In [ ]:
print("="*60)
print("FINAL MODEL EVALUATION ON TEST SET")
print("="*60)

# Evaluate tuned models on test set
final_results = {}

for name, model in tuned_models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)
    
    final_results[name] = {
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'ROC AUC': roc_auc
    }
    
    print(f"\n{name}:")
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall: {recall:.4f}")
    print(f"  F1-Score: {f1:.4f}")
    print(f"  ROC AUC: {roc_auc:.4f}")

final_results_df = pd.DataFrame(final_results).T
display(final_results_df.round(4))

# Show confusion matrices for ALL models first
print("\n" + "="*60)
print("CONFUSION MATRICES FOR ALL MODELS")
print("="*60)

for name, model in tuned_models.items():
    y_pred = model.predict(X_test)
    
    plt.figure(figsize=(6, 5))
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
               xticklabels=['No Disease', 'Disease'], 
               yticklabels=['No Disease', 'Disease'])
    plt.title(f'Confusion Matrix - {name}')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.tight_layout()
    plt.show()

# Select top 4 models based on ROC AUC
top_4_models = final_results_df.nlargest(4, 'ROC AUC')
top_4_model_names = top_4_models.index.tolist()

print("\n" + "="*60)
print("TOP 4 MODELS BASED ON ROC AUC")
print("="*60)
for i, (name, row) in enumerate(top_4_models.iterrows(), 1):
    print(f"{i}. {name}: ROC AUC = {row['ROC AUC']:.4f}")

# Filter tuned_models to only include top 4
top_4_tuned_models = {name: tuned_models[name] for name in top_4_model_names}

# Select best model from top 4
best_model_name = top_4_models.index[0]
best_model = tuned_models[best_model_name]

print(f"\nBest Model: {best_model_name}")
print(f"Best ROC AUC: {top_4_models.loc[best_model_name, 'ROC AUC']:.4f}")

### 5.7 Feature Importance Analysis

In [ ]:
# Feature importance analysis for top 4 models
print("\n" + "="*60)
print("FEATURE IMPORTANCE ANALYSIS - TOP 4 MODELS")
print("="*60)

for name in top_4_model_names:
    model = top_4_tuned_models[name]
    if hasattr(model, 'feature_importances_'):
        feature_importance = pd.DataFrame({
            'feature': X_train.columns,
            'importance': model.feature_importances_
        }).sort_values('importance', ascending=False)
        
        plt.figure(figsize=(10, 8))
        top_features = feature_importance.head(15)
        plt.barh(range(len(top_features)), top_features['importance'])
        plt.yticks(range(len(top_features)), top_features['feature'])
        plt.xlabel('Feature Importance')
        plt.title(f'Feature Importance - {name}')
        plt.gca().invert_yaxis()
        plt.tight_layout()
        plt.show()
        
        print(f"\nTop 10 Most Important Features for {name}:")
        display(feature_importance.head(10))
    else:
        print(f"\n{name}: Feature importance not available for this model type.")

### 5.8 Model Interpretation and Insights

In [ ]:
# Detailed classification reports for top 4 models
print("\n" + "="*60)
print("DETAILED CLASSIFICATION REPORTS - TOP 4 MODELS")
print("="*60)

for name in top_4_model_names:
    model = top_4_tuned_models[name]
    y_pred = model.predict(X_test)
    
    print(f"\n{name}:")
    print("-" * 50)
    print(classification_report(y_test, y_pred, target_names=['No Disease', 'Disease']))



# Final visualization for best model
print("\n" + "="*60)
print(f"DETAILED ANALYSIS - BEST MODEL ({best_model_name})")
print("="*60)

# Decision Tree Visualization
print("\n" + "="*60)
print("DECISION TREE VISUALIZATION")
print("="*60)

# Check if we have a Decision Tree model in our top models
dt_model = None
dt_model_name = None

# Look for Decision Tree in top 4 models
for name in top_4_model_names:
    if 'Decision Tree' in name:
        dt_model = top_4_tuned_models[name]
        dt_model_name = name
        break

# If no Decision Tree in top 4, use the original Decision Tree model
if dt_model is None:
    dt_model = tuned_models.get('Decision Tree')
    dt_model_name = 'Decision Tree'

if dt_model is not None:
    plt.figure(figsize=(20, 12))
    plot_tree(dt_model, max_depth=3, filled=True, feature_names=X_train.columns, 
              class_names=['No Disease', 'Disease'], fontsize=10)
    plt.title(f'Decision Tree Visualization - {dt_model_name}', fontsize=16)
    plt.tight_layout()
    plt.show()
    
    # Show decision tree performance metrics
    y_pred_dt = dt_model.predict(X_test)
    y_prob_dt = dt_model.predict_proba(X_test)[:, 1]
    
    print(f"\nDecision Tree Performance:")
    print(f"  Accuracy: {accuracy_score(y_test, y_pred_dt):.4f}")
    print(f"  Precision: {precision_score(y_test, y_pred_dt):.4f}")
    print(f"  Recall: {recall_score(y_test, y_pred_dt):.4f}")
    print(f"  F1-Score: {f1_score(y_test, y_pred_dt):.4f}")
    print(f"  ROC AUC: {roc_auc_score(y_test, y_prob_dt):.4f}")
else:
    print("No Decision Tree model found in the trained models.")
    print("Please ensure the Decision Tree model was trained successfully.")

### 5.9 Model Persistence

In [ ]:
# Save the best model and results
os.makedirs('../models', exist_ok=True)

# Save best model
joblib.dump(best_model, f'../models/best_model_{best_model_name.lower().replace(" ", "_")}.pkl')

# Save all top 4 models
for name in top_4_model_names:
    model = top_4_tuned_models[name]
    joblib.dump(model, f'../models/top4_model_{name.lower().replace(" ", "_")}.pkl')

# Save all results
final_results_df.to_csv('../models/model_evaluation_results.csv')

# Save top 4 results
top_4_models.to_csv('../models/top_4_models_results.csv')

print("="*60)
print("MODEL PERSISTENCE")
print("="*60)
print(f"Best model saved: ../models/best_model_{best_model_name.lower().replace(' ', '_')}.pkl")
print("Top 4 models saved:")
for name in top_4_model_names:
    print(f"  - ../models/top4_model_{name.lower().replace(' ', '_')}.pkl")
print("All evaluation results saved: ../models/model_evaluation_results.csv")
print("Top 4 results saved: ../models/top_4_models_results.csv")

### Modeling and Evaluation Summary

The modeling and evaluation phase included:

1. **Model Selection**: Evaluated 9 different algorithms including tree-based models, linear models, and ensemble methods.

2. **Initial Evaluation**: Compared models using accuracy, precision, recall, F1-score, and ROC AUC metrics.

3. **Hyperparameter Tuning**: Applied GridSearchCV to optimize the top-performing models.

4. **Cross-Validation**: Used 5-fold cross-validation to assess model generalization.

5. **Overfitting Analysis**: Analyzed decision tree performance across different depths to identify optimal complexity.

6. **Comprehensive Confusion Matrix Analysis**: Displayed confusion matrices for all models to understand classification patterns.

7. **Top 4 Model Selection**: Selected the top 4 models based on ROC AUC scores for detailed analysis.

8. **Feature Importance Analysis**: Analyzed feature importance for all top 4 models to understand key predictors.

9. **Comparative Analysis**: Generated ROC curves and Precision-Recall curves for top 4 models.

10. **Detailed Classification Reports**: Provided comprehensive classification reports for all top 4 models.

11. **Model Persistence**: Saved the best-performing model and all top 4 models for future use.

The analysis focused on the top 4 performing models after displaying all original confusion matrices, providing a comprehensive evaluation of cardiovascular disease prediction models.